# Final answer: retrieval evidence → Qwen3.5-9B

Place `retrieval_manifest.jsonl` and `data/tables/` under `/content/answer_inputs`, select an A100 runtime, then **Run all**. Qwen selects cells and a small expression, Python validates/executes it, and packaging is refused until all 1,012 answers pass.

In [ ]:
%pip uninstall -y torchaudio
%pip install -q 'pydantic>=2.7' 'openai>=1.100' 'vllm==0.28.0' 'transformers>=5.0.0' pandas

In [ ]:
from __future__ import annotations

import ast
import asyncio
import hashlib
import json
import os
import re
import subprocess
import time
import zipfile
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI
from pydantic import BaseModel, Field, model_validator

MODEL = 'Qwen/Qwen3.5-9B'
REVISION = 'c202236235762e1c871ad0ccb60c8ee5ba337b9a'
EXPECTED_RETRIEVAL_SHA256 = os.environ.get('RETRIEVAL_SHA256', '')
ROOT = Path('/content')
INPUT = Path(os.environ.get('ANSWER_INPUT_DIR', str(ROOT / 'answer_inputs')))
RUN = Path(os.environ.get('ANSWER_RUN_DIR', str(ROOT / 'qwen35_answer')))
CHECKPOINT = RUN / 'answer_programs_qwen35.jsonl'
AUDIT = RUN / 'answer_audit_qwen35.jsonl'
OUTPUT_ZIP = Path(os.environ.get('ANSWER_OUTPUT_ZIP', str(ROOT / 'submission_qwen35_grounded.zip')))
LIMIT = 0                 # 0 = all 1,012 questions
CONCURRENCY = 64
MAX_ROWS_PER_TABLE = 14
MAX_CONTEXT_CHARS = 30_000
MAX_REPAIRS = 3
PORT = 8011

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

retrieval_path = INPUT / 'retrieval_manifest.jsonl'
if not retrieval_path.is_file():
    raise FileNotFoundError(f'Missing {retrieval_path}')
actual_retrieval_sha256 = sha256(retrieval_path)
if EXPECTED_RETRIEVAL_SHA256 and actual_retrieval_sha256 != EXPECTED_RETRIEVAL_SHA256:
    raise ValueError(f'Retrieval hash mismatch: {actual_retrieval_sha256}')
EXPECTED_RETRIEVAL_SHA256 = actual_retrieval_sha256
RUN.mkdir(parents=True, exist_ok=True)

retrieval_rows = [json.loads(line) for line in retrieval_path.read_text(encoding='utf-8').splitlines() if line.strip()]
submission_path = RUN / 'submission.json'
if len(retrieval_rows) != 1012 or len({int(r['id']) for r in retrieval_rows}) != 1012:
    raise ValueError('Expected 1,012 unique submission rows')
for row in retrieval_rows:
    for evidence in row['evidence']:
        if not (INPUT / evidence['csv_path']).is_file():
            raise FileNotFoundError(evidence['csv_path'])
print({'retrieval_sha256': sha256(retrieval_path), 'questions': len(retrieval_rows), 'evidence_csvs': sum(len(r['evidence']) for r in retrieval_rows)})

server_log = (RUN / 'vllm.log').open('w', encoding='utf-8')
server = subprocess.Popen([
    'vllm', 'serve', MODEL,
    '--revision', REVISION,
    '--trust-remote-code',
    '--language-model-only',
    '--performance-mode', 'throughput',
    '-O1',
    '--generation-config', 'vllm',
    '--reasoning-parser', 'qwen3',
    '--enable-prefix-caching',
    '--served-model-name', MODEL,
    '--host', '127.0.0.1',
    '--port', str(PORT),
    '--dtype', 'bfloat16',
    '--max-model-len', '16384',
    '--max-num-seqs', '128',
    '--max-num-batched-tokens', '65536',
    '--gpu-memory-utilization', '0.94',
], stdout=server_log, stderr=subprocess.STDOUT)

client = AsyncOpenAI(base_url=f'http://127.0.0.1:{PORT}/v1', api_key='local')
deadline = time.time() + 900
while True:
    try:
        await client.models.list()
        break
    except Exception:
        if server.poll() is not None:
            server_log.flush()
            raise RuntimeError((RUN / 'vllm.log').read_text(encoding='utf-8')[-6000:])
        if time.time() > deadline:
            raise TimeoutError('vLLM startup timed out')
        await asyncio.sleep(0.25)
print('vLLM ready')

## Grounded program contract

Numbers are masked from the model. It may choose visible `(dataframe, row, column)` coordinates and compose only a small validated arithmetic expression.

In [ ]:
class CellFact(BaseModel):
    id: str = Field(pattern=r'^f\d+$')
    variable: str = Field(pattern=r'^df\d+$')
    row: int = Field(ge=0)
    column: int = Field(ge=0)
    metric: str = Field(min_length=2, max_length=240)

class EvidenceProgram(BaseModel):
    cells: list[CellFact] = Field(min_length=1, max_length=64)
    expression: str = Field(min_length=1, max_length=4000)

    @model_validator(mode='after')
    def unique_ids(self):
        ids = [cell.id for cell in self.cells]
        if len(ids) != len(set(ids)):
            raise ValueError('duplicate cell IDs')
        return self

NUMBER = re.compile(r'(?<![\w])[-+]?\(?\d[\d., ]*\)?%?(?![\w])')
YEAR = re.compile(r'^(?:19|20)\d{2}$')
WORD = re.compile(r'[a-zA-ZÀ-ỹ0-9]+')
ALLOWED_CALLS = {'abs', 'min', 'max', 'sum', 'mean', 'median', 'ratio_pct', 'pct_change', 'select_max', 'select_min'}
ALLOWED_NODES = (ast.Expression, ast.Name, ast.Load, ast.Constant, ast.List, ast.Tuple, ast.BinOp, ast.UnaryOp, ast.BoolOp, ast.Compare, ast.IfExp, ast.Call, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Mod, ast.Pow, ast.USub, ast.UAdd, ast.And, ast.Or, ast.Eq, ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE)

def normalize(text: object) -> str:
    import unicodedata
    value = unicodedata.normalize('NFD', str(text).lower().replace('đ', 'd'))
    return ' '.join(re.findall(r'[a-z0-9]+', ''.join(c for c in value if unicodedata.category(c) != 'Mn')))

def mask(value: object) -> str:
    text = str(value)
    if YEAR.fullmatch(text.strip()):
        return text
    return NUMBER.sub('<NUM>', text)

def load_frames(row: dict) -> dict[str, pd.DataFrame]:
    return {e['variable']: pd.read_csv(INPUT / e['csv_path'], dtype=str, keep_default_na=False) for e in row['evidence']}

def selected_rows(question: str, frame: pd.DataFrame) -> list[int]:
    query_tokens = set(normalize(question).split()) - {'la', 'cua', 'va', 'nam', 'bao', 'nhieu'}
    scored = []
    for index, values in frame.iterrows():
        tokens = set(normalize(' '.join(map(str, values.tolist()))).split())
        scored.append((len(query_tokens & tokens), index))
    seeds = [index for _, index in sorted(scored, reverse=True)[:10]] + list(range(min(3, len(frame))))
    rows = []
    for index in seeds:
        for near in (index - 1, index, index + 1):
            if 0 <= near < len(frame) and near not in rows:
                rows.append(near)
    return sorted(rows[:MAX_ROWS_PER_TABLE])

def context_for(row: dict, frames: dict[str, pd.DataFrame]) -> tuple[str, dict[str, set[int]]]:
    table_ids = row.get('relevant_tables', [])
    selected = []
    for position, evidence in enumerate(row['evidence']):
        variable = evidence['variable']
        frame = frames[variable]
        title = table_ids[position] if position < len(table_ids) else evidence['csv_path']
        selected.append((variable, title, frame, selected_rows(row['question'], frame)))
    row_limit = MAX_ROWS_PER_TABLE
    while True:
        blocks, visible = [], {}
        for variable, title, frame, rows in selected:
            rows = rows[:row_limit]
            visible[variable] = set(rows)
            lines = [f'TABLE {variable} | {title}', 'COLUMNS ' + ' | '.join(f'{i}:{mask(c)}' for i, c in enumerate(frame.columns))]
            lines.extend(f'ROW {i} | ' + ' | '.join(f'{j}:{mask(value)}' for j, value in enumerate(frame.iloc[i].tolist())) for i in rows)
            blocks.append('\n'.join(lines))
        context = '\n\n'.join(blocks)
        if len(context) <= MAX_CONTEXT_CHARS or row_limit <= 2:
            return context, visible
        row_limit = max(2, row_limit - 2)

def parse_number(value: object) -> float:
    text = str(value).strip().replace(' ', '')
    if not text or text.lower() in {'nan', 'none', '-', '--'}:
        raise ValueError(f'not numeric: {value!r}')
    negative = text.startswith('(') and text.endswith(')')
    percent = text.endswith('%')
    text = text.strip('()%').replace('−', '-').replace('–', '-')
    if ',' in text and '.' in text:
        if text.rfind(',') > text.rfind('.'):
            text = text.replace('.', '').replace(',', '.')
        else:
            text = text.replace(',', '')
    elif ',' in text:
        tail = text.rsplit(',', 1)[1]
        text = text.replace(',', '.') if len(tail) <= 2 else text.replace(',', '')
    elif text.count('.') > 1 or (text.count('.') == 1 and len(text.rsplit('.', 1)[1]) == 3):
        text = text.replace('.', '')
    number = float(text)
    if negative:
        number = -abs(number)
    return number / 100 if percent else number

def source_scale(frame: pd.DataFrame) -> float:
    sample = normalize(' '.join(map(str, frame.columns)) + ' ' + ' '.join(map(str, frame.head(4).to_numpy().ravel())))
    if 'nghin ty' in sample:
        return 1e9
    if 'ty dong' in sample or 'don vi ty' in sample:
        return 1e9
    if 'trieu dong' in sample or 'don vi trieu' in sample:
        return 1e6
    if 'nghin dong' in sample or 'don vi nghin' in sample:
        return 1e3
    return 1.0

def target_scale(question: str) -> float:
    text = normalize(question)
    if 'nghin ty dong' in text:
        return 1e9
    if 'ty dong' in text:
        return 1e9
    if 'trieu dong' in text:
        return 1e6
    if 'nghin dong' in text:
        return 1e3
    return 1.0

def validate_expression(program: EvidenceProgram, question: str) -> ast.Expression:
    tree = ast.parse(program.expression, mode='eval')
    fact_ids = {cell.id for cell in program.cells}
    question_numbers = {float(x.replace(',', '.')) for x in re.findall(r'(?<!\d)\d+(?:[.,]\d+)?', question)}
    permitted_numbers = question_numbers | {0.0, 1.0, 100.0}
    used = set()
    for node in ast.walk(tree):
        if not isinstance(node, ALLOWED_NODES):
            raise ValueError(f'forbidden expression node: {type(node).__name__}')
        if isinstance(node, ast.Name):
            if node.id not in fact_ids and node.id not in ALLOWED_CALLS:
                raise ValueError(f'unknown name: {node.id}')
            if node.id in fact_ids:
                used.add(node.id)
        if isinstance(node, ast.Call):
            if not isinstance(node.func, ast.Name) or node.func.id not in ALLOWED_CALLS:
                raise ValueError('forbidden function')
            arity = {'abs': {1}, 'mean': set(range(1, 65)), 'median': {1}, 'ratio_pct': {2}, 'pct_change': {2}, 'select_max': {1, 2}, 'select_min': {1, 2}}
            if node.func.id in arity and len(node.args) not in arity[node.func.id]:
                raise ValueError(f'wrong argument count for {node.func.id}')
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and float(node.value) not in permitted_numbers:
            raise ValueError(f'numeric literal not grounded in question: {node.value}')
    if used != fact_ids:
        raise ValueError(f'unused cells: {sorted(fact_ids - used)}')
    return tree

def compile_node(node: ast.AST, cells: dict[str, CellFact], frames: dict[str, pd.DataFrame]) -> str:
    if isinstance(node, ast.Name):
        cell = cells[node.id]
        scale = source_scale(frames[cell.variable])
        return f'(parse_number({cell.variable}.iloc[{cell.row}, {cell.column}]) * {scale!r})'
    if isinstance(node, ast.Constant):
        return repr(node.value)
    if isinstance(node, ast.List):
        return '[' + ', '.join(compile_node(x, cells, frames) for x in node.elts) + ']'
    if isinstance(node, ast.Tuple):
        return '(' + ', '.join(compile_node(x, cells, frames) for x in node.elts) + ',)'
    if isinstance(node, ast.UnaryOp):
        op = '-' if isinstance(node.op, ast.USub) else '+'
        return f'({op}{compile_node(node.operand, cells, frames)})'
    if isinstance(node, ast.BinOp):
        symbol = {ast.Add: '+', ast.Sub: '-', ast.Mult: '*', ast.Div: '/', ast.Mod: '%', ast.Pow: '**'}[type(node.op)]
        return f'({compile_node(node.left, cells, frames)} {symbol} {compile_node(node.right, cells, frames)})'
    if isinstance(node, ast.Compare):
        symbols = {ast.Eq: '==', ast.NotEq: '!=', ast.Lt: '<', ast.LtE: '<=', ast.Gt: '>', ast.GtE: '>='}
        parts = [compile_node(node.left, cells, frames)]
        for op, value in zip(node.ops, node.comparators):
            parts.extend((symbols[type(op)], compile_node(value, cells, frames)))
        return '(' + ' '.join(parts) + ')'
    if isinstance(node, ast.BoolOp):
        joiner = ' and ' if isinstance(node.op, ast.And) else ' or '
        return '(' + joiner.join(compile_node(x, cells, frames) for x in node.values) + ')'
    if isinstance(node, ast.IfExp):
        return f'({compile_node(node.body, cells, frames)} if {compile_node(node.test, cells, frames)} else {compile_node(node.orelse, cells, frames)})'
    if isinstance(node, ast.Call):
        name = node.func.id
        args = [compile_node(x, cells, frames) for x in node.args]
        if name == 'mean':
            values = args[0] if len(args) == 1 else '[' + ', '.join(args) + ']'
            return f'(sum({values}) / len({values}))'
        if name == 'median':
            return f'float(pd.Series({args[0]}).median())'
        if name == 'ratio_pct':
            return f'(({args[0]}) / ({args[1]}) * 100.0)'
        if name == 'pct_change':
            return f'((({args[0]}) - ({args[1]})) / abs({args[1]}) * 100.0)'
        if name in {'select_max', 'select_min'} and len(args) == 1:
            chooser = 'max' if name == 'select_max' else 'min'
            return f'{chooser}({args[0]})'
        if name in {'select_max', 'select_min'}:
            chooser = 'max' if name == 'select_max' else 'min'
            return f'({args[1]})[({args[0]}).index({chooser}({args[0]}))]'
        return f'{name}(' + ', '.join(args) + ')'
    raise ValueError(f'cannot compile {type(node).__name__}')

def compile_program(program: EvidenceProgram, question: str, frames: dict[str, pd.DataFrame], visible: dict[str, set[int]]) -> str:
    tree = validate_expression(program, question)
    cells = {cell.id: cell for cell in program.cells}
    for cell in program.cells:
        if cell.variable not in frames:
            raise ValueError(f'unknown dataframe: {cell.variable}')
        frame = frames[cell.variable]
        if cell.row not in visible.get(cell.variable, set()):
            raise ValueError(f'row was not shown: {cell.variable}[{cell.row}]')
        if cell.row >= len(frame) or cell.column >= len(frame.columns):
            raise ValueError(f'cell out of range: {cell.variable}[{cell.row},{cell.column}]')
        parse_number(frame.iloc[cell.row, cell.column])
    expression = compile_node(tree.body, cells, frames)
    return f'result = round(float(({expression}) / {target_scale(question)!r}), 2)'

SAFE_GLOBALS = {'__builtins__': {'abs': abs, 'min': min, 'max': max, 'sum': sum, 'len': len, 'float': float, 'round': round}, 'pd': pd, 'parse_number': parse_number}

def replay(query: str, frames: dict[str, pd.DataFrame]) -> float:
    scope = dict(frames)
    exec(compile(query, '<grounded-query>', 'exec'), SAFE_GLOBALS, scope)
    result = float(scope['result'])
    if not (-1e30 < result < 1e30):
        raise ValueError('non-finite or implausibly large result')
    return result

def prompt_for(row: dict, context: str, previous: str | None = None, error: str | None = None) -> str:
    repair = '' if previous is None else f'\nPREVIOUS INVALID JSON:\n{previous[:6000]}\nVALIDATION ERROR:\n{error}\nCorrect only this failure.\n'
    return f'''You produce one grounded numerical program for a Vietnamese financial question.

Do not answer directly. Do not return Python. Select only cells shown below. Numbers are masked, so never copy or invent a financial value.

Return JSON only:
{{"cells":[{{"id":"f1","variable":"df0","row":3,"column":2,"metric":"complete accounting concept"}}],"expression":"f1"}}

Expression grammar: fact IDs, + - * /, comparisons, conditionals, and only these functions:
abs, min, max, sum, mean, median, ratio_pct(numerator, denominator), pct_change(current, previous), select_max(scores, values), select_min(scores, values).

Use the exact requested year, company, scope, total/component, opening/closing, gross/net, counterparty, and unit. For selector questions, bind every candidate's selector operands and downstream values. Use each selected cell in the expression.

QUESTION:
{row['question']}
{repair}
EVIDENCE:
{context}
'''

# Small local contract check.
_test_frame = pd.DataFrame({'Năm 2024': ['1.234']})
_test = EvidenceProgram.model_validate({'cells': [{'id': 'f1', 'variable': 'df0', 'row': 0, 'column': 0, 'metric': 'Tiền'}], 'expression': 'f1'})
_query = compile_program(_test, 'Bao nhiêu đồng năm 2024?', {'df0': _test_frame}, {'df0': {0}})
assert replay(_query, {'df0': _test_frame}) == 1234.0
print('grounded contract: PASS')

In [ ]:
async def generate(prompt: str) -> str:
    response = await client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=1536,
        response_format={'type': 'json_object'},
        extra_body={'chat_template_kwargs': {'enable_thinking': False}},
    )
    content = response.choices[0].message.content
    if not content:
        raise ValueError('model returned no content')
    return content

async def answer_one(row: dict) -> dict:
    try:
        frames = load_frames(row)
        context, visible = context_for(row, frames)
    except Exception as exc:
        return {'id': int(row['id']), 'status': 'failed', 'attempts': 0, 'errors': [f'PREPARE: {type(exc).__name__}: {exc}']}
    previous = error = None
    errors = []
    for attempt in range(1, MAX_REPAIRS + 2):
        try:
            raw = await generate(prompt_for(row, context, previous, error))
            program = EvidenceProgram.model_validate_json(raw)
            query = compile_program(program, row['question'], frames, visible)
            answer = replay(query, frames)
            return {'id': int(row['id']), 'status': 'accepted', 'attempts': attempt, 'answer': answer, 'pandas_query': query, 'program': program.model_dump(), 'errors': errors}
        except Exception as exc:
            previous = locals().get('raw', '')
            error = f'{type(exc).__name__}: {exc}'
            errors.append(error)
    return {'id': int(row['id']), 'status': 'failed', 'attempts': MAX_REPAIRS + 1, 'errors': errors}

config = {'model': MODEL, 'revision': REVISION, 'retrieval_sha256': EXPECTED_RETRIEVAL_SHA256, 'max_repairs': MAX_REPAIRS, 'max_rows_per_table': MAX_ROWS_PER_TABLE, 'max_context_chars': MAX_CONTEXT_CHARS}
config_sha = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
config_path = RUN / 'answer_config.json'
if config_path.exists() and json.loads(config_path.read_text())['config_sha256'] != config_sha:
    raise ValueError('Checkpoint configuration changed; use a new RUN directory')
config_path.write_text(json.dumps({'config_sha256': config_sha, **config}, indent=2), encoding='utf-8')
completed, latest = {}, {}
if CHECKPOINT.exists():
    for line in CHECKPOINT.read_text(encoding='utf-8').splitlines():
        record = json.loads(line)
        latest[int(record['id'])] = record
        if record['status'] == 'accepted':
            completed[int(record['id'])] = record

targets = retrieval_rows[:LIMIT or None]
pending = [row for row in targets if int(row['id']) not in completed]
started = time.time()
for offset in range(0, len(pending), CONCURRENCY):
    batch = pending[offset:offset + CONCURRENCY]
    results = await asyncio.gather(*(answer_one(row) for row in batch))
    with CHECKPOINT.open('a', encoding='utf-8') as handle:
        for result in results:
            handle.write(json.dumps(result, ensure_ascii=False) + '\n')
            handle.flush()
            latest[result['id']] = result
            if result['status'] == 'accepted':
                completed[result['id']] = result
    done = len(targets) - len([row for row in targets if int(row['id']) not in completed])
    elapsed = max(time.time() - started, 1e-9)
    print(f'{done}/{len(targets)} accepted | {done / elapsed:.2f} questions/s | failed_this_batch={sum(r["status"] == "failed" for r in results)}', flush=True)
remaining = [int(row['id']) for row in targets if int(row['id']) not in completed]
print({'accepted': len(completed), 'remaining': len(remaining), 'failed_ids': remaining[:30]})

In [ ]:
audit_rows = [{'id': qid, 'status': result['status'], 'attempts': result['attempts'], 'errors': result['errors']} for qid, result in latest.items()]
AUDIT.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in sorted(audit_rows, key=lambda x: x['id'])), encoding='utf-8')
missing = [int(row['id']) for row in retrieval_rows if int(row['id']) not in completed]
if LIMIT or missing:
    raise RuntimeError(f'Not packaging: LIMIT={LIMIT}, unresolved={missing[:30]}. Rerun the generation cell to retry only failures.')

by_id = {int(row['id']): row for row in retrieval_rows}
retrieval_keys = ('relevant_docs', 'relevant_tables', 'evidence')
for qid, result in completed.items():
    row = by_id[qid]
    before = {key: json.dumps(row[key], ensure_ascii=False, sort_keys=True) for key in retrieval_keys}
    row['answer'] = result['answer']
    row['pandas_query'] = result['pandas_query']
    replayed = replay(row['pandas_query'], load_frames(row))
    if abs(replayed - float(row['answer'])) > 1e-9:
        raise ValueError(f'replay mismatch for Q{qid}')
    if any(before[key] != json.dumps(row[key], ensure_ascii=False, sort_keys=True) for key in retrieval_keys):
        raise ValueError(f'retrieval mutated for Q{qid}')

submission_path.write_text(json.dumps(retrieval_rows, ensure_ascii=False, indent=2), encoding='utf-8')

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    archive.write(submission_path, 'submission.json')
    for name in sorted({item['csv_path'] for row in retrieval_rows for item in row['evidence']}):
        archive.write(INPUT / name, name)
with zipfile.ZipFile(OUTPUT_ZIP) as archive:
    bad_member = archive.testzip()
    packaged_rows = json.loads(archive.read('submission.json'))
    packaged_names = set(archive.namelist())
if bad_member or len(packaged_rows) != 1012:
    raise ValueError(f'bad output package: member={bad_member}, rows={len(packaged_rows)}')
for row in packaged_rows:
    if not isinstance(row['answer'], (int, float)) or not isinstance(row['pandas_query'], str):
        raise ValueError(f'invalid answer row Q{row["id"]}')
    for evidence in row['evidence']:
        if evidence['csv_path'] not in packaged_names:
            raise ValueError(f'missing evidence: {evidence["csv_path"]}')

status_counts = pd.Series([r['status'] for r in latest.values()]).value_counts().to_dict()
manifest = {
    'retrieval_manifest_sha256': EXPECTED_RETRIEVAL_SHA256,
    'output_zip_sha256': sha256(OUTPUT_ZIP),
    'model': MODEL,
    'revision': REVISION,
    'questions_total': len(retrieval_rows),
    'questions_attempted': len(latest),
    'status_counts': status_counts,
    'retrieval_unchanged': True,
    'zip_integrity': True,
    'evidence_files': len(packaged_names) - 1,
}
(RUN / 'answer_manifest_qwen35.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
print(OUTPUT_ZIP)

server.terminate()
server.wait(timeout=30)
server_log.close()

In [ ]:
from google.colab import files
files.download(str(OUTPUT_ZIP))
files.download(str(RUN / 'answer_manifest_qwen35.json'))
files.download(str(AUDIT))
files.download(str(CHECKPOINT))